In [17]:
# =====================================================
# AI DIET RECOMMENDATION SYSTEM
# =====================================================

import pandas as pd

import joblib

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv(
    r"/home/cnssec/Downloads/diet/Cleaned_data/feature_engineered_dataset.csv"
)

# =====================================================
# LOAD TRAINED MODEL
# =====================================================

model = joblib.load(
    r"/home/cnssec/Downloads/diet/Cleaned_data/diet_recommendation_model.pkl"
)

In [18]:
# =====================================================
# FOOD RECOMMENDATION FUNCTION
# =====================================================

def recommend_foods(
    gender,
    age,
    height,
    weight,
    meals_per_day,
    bmr
):

    # =================================================
    # BMI CALCULATION
    # =================================================

    height_m = height / 100

    bmi = weight / (height_m ** 2)

    # =================================================
    # DAILY REQUIREMENTS
    # =================================================

    water_intake = round(weight * 0.033, 2)

    protein_need = round(weight * 1.6, 2)

    # =================================================
    # MODEL INPUT
    # =================================================

    input_data = pd.DataFrame({

        'gender': [gender],

        'height': [height],

        'weight': [weight],

        'age': [age],

        'bmi': [bmi],

        'meals_per_day': [meals_per_day],

        'bmr': [bmr],

        'water_intake_liters': [water_intake],

        'daily_protein_need': [protein_need]

    })

    # =================================================
    # MODEL PREDICTION
    # =================================================

    prediction = model.predict(input_data)[0]

    # =================================================
    # RECOMMENDATION LOGIC
    # =================================================

    # IMPORTANT:
    # CHECK YOUR LABEL ENCODING VALUES
    # 0 = Weight Gain
    # 1 = Maintenance
    # 2 = Weight Loss

    if prediction == 2:

        # Weight Loss

        recommendations = df[
            (df['calories'] > 50) &
            (df['calories'] < 500)
        ]

        goal = "Weight Loss"

    elif prediction == 1:

        # Maintenance

        recommendations = df[
            (df['calories'] >= 300) &
            (df['calories'] <= 800)
        ]

        goal = "Maintenance"

    else:

        # Weight Gain

        recommendations = df[
            (df['calories'] > 500)
        ]

        goal = "Weight Gain"

    # =================================================
    # REMOVE DUPLICATE FOODS
    # =================================================

    recommendations = recommendations.drop_duplicates(
        subset=['food']
    )
    recommendations = recommendations.sample(
        min(10, len(recommendations))
    )
    # =================================================
    # KEEP IMPORTANT COLUMNS
    # =================================================

    recommendations = recommendations[
        [
            'food',
            'calories',
            'protein',
            'carbs',
            'fat',
            'fiber'
        ]
    ]

    # =================================================
    # TOP 10 RECOMMENDATIONS
    # =================================================

    recommendations = recommendations.head(10)

    # =================================================
    # RETURN RESULTS
    # =================================================

    return (
        goal,
        round(bmi, 2),
        water_intake,
        protein_need,
        recommendations
    )

In [19]:
# =====================================================
# TESTING
# =====================================================

if __name__ == "__main__":

    goal, bmi, water, protein, foods = recommend_foods(

        gender=1,          # Male = 1, Female = 0

        age=24,

        height=175,

        weight=72,

        meals_per_day=4,

        bmr=1800
    )

    print("\n==============================")
    print("AI DIET RECOMMENDATION SYSTEM")
    print("==============================")

    print("\nPredicted Goal:")
    print(goal)

    print("\nBMI:")
    print(bmi)

    print("\nDaily Water Intake:")
    print(water, "Liters")

    print("\nDaily Protein Need:")
    print(protein, "grams")

    print("\nRecommended Foods:\n")

    print(foods)


AI DIET RECOMMENDATION SYSTEM

Predicted Goal:
Weight Gain

BMI:
23.51

Daily Water Intake:
2.38 Liters

Daily Protein Need:
115.2 grams

Recommended Foods:

                                                   food  calories  protein  \
499            Beans, Dry, Navy, 11F-8107 (0% moisture)  1729.502    24.40   
294   Proximates, Yogurt, Greek, strawberry, non-fat...  1951.502     8.36   
1310  Proximates, Greek yogurt, CHOBANI PLAIN NON-FA...  1561.800     0.00   
388   Proximates, Greek yogurt, FAGE PLAIN NONFAT (O...  1576.800     0.00   
1043                         Lunchmeat, ham - NFY1210Q3  1716.000     0.00   
1885    Proximates, Tamale, Pork (TX-HA,NV) - NFY0902AA  1807.800     0.00   
440            Beans, Dry, Small Red, 797 (0% moisture)  2230.941    20.80   
378   Total Fat, Ground turkey, 93% lean, 7% fat, pa...  2504.218     0.00   
549   Proximates, Onion rings, frozen, ORE-IDA (CA2,...  2936.818     0.00   
429                Beans, Dry, Pinto, 821 (0% moisture)  2523